# Autoencoder Anomaly Detection Practice

**Topic:** Autoencoders and Representation Learning &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(5000, 32)`: normal samples only, generated by a smooth *nonlinear* map from an 8-D latent space.
- `X_test` `(2000, 32)`: a mix of normal samples and ~8% anomalies. The evaluation cell scores you against the label array `_y_anom` built in the data cell above.
- Because the manifold is nonlinear, a nonlinear autoencoder (PyTorch or sklearn, bottleneck ≈ 8) beats plain PCA, but a good PCA pipeline can still pass.

## Your task

1. Train an autoencoder on `X_train` (e.g., 32→16→8→16→32 with ReLU, MSE loss, Adam). PyTorch or sklearn will both do.
2. Compute per-sample reconstruction error on `X_test`.
3. Set `anomaly_scores` to a `(2000,)` array where **larger = more anomalous**.

## Evaluation

ROC-AUC of your scores against the label array `_y_anom` must be ≥ 0.85.


In [ ]:
import numpy as np

rng = np.random.default_rng(331)
d, k_lat = 32, 8
W1 = rng.normal(0, 1.0, (k_lat, 20))
W2 = rng.normal(0, 1.0, (20, d))

def normal_batch(n, seed):
    r = np.random.default_rng(seed)
    z = r.normal(size=(n, k_lat))
    return np.tanh(z @ W1) @ W2 + r.normal(0, 0.15, (n, d))

X_train = normal_batch(5000, 341)

_n_test, _n_anom = 2000, 160
_normal = normal_batch(_n_test - _n_anom, 342)
_anom = rng.normal(0, 1.0, (_n_anom, d)) * np.abs(normal_batch(1, 343)).mean() * 2.2
X_test = np.vstack([_normal, _anom])
_y_anom = np.concatenate([np.zeros(_n_test - _n_anom, int), np.ones(_n_anom, int)])
_shuf = rng.permutation(_n_test)
X_test, _y_anom = X_test[_shuf], _y_anom[_shuf]

print(f"X_train {X_train.shape} (all normal), X_test {X_test.shape} (~8% anomalies)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: ROC-AUC of your anomaly scores
# =====================================================================
import numpy as np
from sklearn.metrics import roc_auc_score

if "anomaly_scores" not in dir():
    raise NameError("Define `anomaly_scores` in the solution cell before running this cell.")

_s = np.asarray(anomaly_scores, dtype=float).ravel()
if _s.shape != _y_anom.shape:
    raise ValueError(f"`anomaly_scores` has shape {_s.shape}, expected {_y_anom.shape}.")

_auc = float(roc_auc_score(_y_anom, _s))
print("=" * 52)
print(f"ROC-AUC : {_auc:.4f}   (target: >= 0.85, 0.5 = random guessing)")
print("=" * 52)
print("PASSED: your representation captured the manifold!" if _auc >= 0.85 else "NOT PASSED yet: train longer or shrink the bottleneck.")


## Solution

The manifold is nonlinear, so mirror it with a nonlinear autoencoder: sklearn's MLPRegressor trained to reproduce its own input through a narrow bottleneck (32-16-8-16-32). Reconstruction error on X_test is the anomaly score.

```python
import numpy as np
from sklearn.neural_network import MLPRegressor

ae = MLPRegressor(hidden_layer_sizes=(16, 8, 16), activation="relu",
                  max_iter=800, random_state=0)
ae.fit(X_train, X_train)                     # learn to reconstruct normal data

recon = ae.predict(X_test)
anomaly_scores = np.linalg.norm(X_test - recon, axis=1)
print(f"score range: [{anomaly_scores.min():.2f}, {anomaly_scores.max():.2f}]")
```
